In [1]:
import pickle   # importing pickle for saving and loading machine learning models
import pandas as pd  # importing pandas for analyzing, cleaning, exploring, and manipulating data
from sklearn.model_selection import train_test_split  # importing train_test_split for spliting the data
#from preprocess1 import *  # importing * for import all functions at once
from imblearn.over_sampling import SMOTE  # importing SMOTE for Balancing the Data

In [2]:
df=pd.read_csv(r"C:\Users\mails\OneDrive\Documents\Python\Datamites\SVM_updated\loan_approved.csv")  # Loading dataset

In [3]:
df.isnull().sum()

Loan_ID                    0
Gender                    13
Married                    3
Dependents                15
Education                  0
Self_Employed             32
ApplicantIncome            0
CoapplicantIncome          0
LoanAmount                22
Loan_Amount_Term          14
Credit_History            50
Property_Area              0
Loan_Status (Approved)     0
dtype: int64

In [6]:
df.loc[df['Gender'].isnull(),'Gender']="Male"
df.loc[df['Married'].isnull(),'Married']="Yes"
df.loc[df['Dependents'].isnull(),'Dependents']='0'  
df.loc[df['Self_Employed'].isnull(),'Self_Employed']='No' 
df.loc[df['LoanAmount'].isnull(),'LoanAmount']=df['LoanAmount'].median()
df.loc[df['Loan_Amount_Term'].isnull(),'Loan_Amount_Term']=360.0
df.loc[df['Credit_History'].isnull(),'Credit_History']=0.0

In [11]:
import sys

sys.path.append(
    r"C:\Users\mails\OneDrive\Documents\Python\Datamites\SVM_updated"
)

In [12]:
import pickle
import __main__

from preprocess1 import (
    ModifiedLabelEncoder,
    divide_by_12,
    same
)

# Make the custom objects available in __main__
__main__.ModifiedLabelEncoder = ModifiedLabelEncoder
__main__.divide_by_12 = divide_by_12
__main__.same = same

# Load preprocessor
with open(
    r"C:\Users\mails\OneDrive\Documents\Python\Datamites\SVM_updated\preprocessing.pkl",
    "rb"
) as f:
    preprocessor = pickle.load(f)

print("Preprocessor loaded successfully!")

Preprocessor loaded successfully!


In [13]:
preprocessor

,transformers,"[('OHE columns', ...), ('Label_encoder', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,True
,categories,'auto'
,drop,None
,sparse_output,True


In [18]:
preprocessor

,transformers,"[('OHE columns', ...), ('Label_encoder', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,True
,categories,'auto'
,drop,None
,sparse_output,True


In [15]:
# Spliting the data into train and test
train,test,_,_=train_test_split(df,df['Loan_Status (Approved)'],test_size=0.2)

In [21]:
df[0:1]

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status (Approved)
0,LP001002,Male,No,0,Graduate,No,5849,0.0,128.0,360.0,1.0,Urban,Y


In [16]:
# Transform the training data using the preprocessor object or PipeLine
processed_data=preprocessor.fit_transform(train) #We are applying preprocessing step for train data

In [19]:
processed_data

array([[1., 0., 0., ..., 1., 1., 1.],
       [1., 0., 0., ..., 1., 1., 0.],
       [0., 1., 0., ..., 0., 1., 1.],
       ...,
       [1., 0., 0., ..., 2., 0., 1.],
       [1., 0., 0., ..., 0., 1., 1.],
       [1., 0., 1., ..., 0., 1., 1.]])

In [22]:
y_train=processed_data[:,-1]
x_train=processed_data[:,:-1]

In [23]:
smote=SMOTE()
x_train_smote,y_train_smote=smote.fit_resample(x_train,y_train)

In [24]:
x_train.shape

(491, 15)

In [26]:
x_train_smote.shape

(674, 15)

In [27]:
df['Loan_Status (Approved)'].value_counts()

Loan_Status (Approved)
Y    422
N    192
Name: count, dtype: int64

In [31]:
test_processed=preprocessor.transform(test) 
x_test=test_processed[:,:-1]  # Extract the features (all columns except the last one) from the processed data
y_test=test_processed[:,-1] 

In [32]:
from sklearn.svm import SVC  # # assign Support vector classifier
svclassifier = SVC() ## base model with default parameters
svclassifier.fit(x_train_smote,y_train_smote)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [33]:
y_pred=svclassifier.predict(x_test)

In [34]:
# Importing the classification_report function from sklearn.metrics
from sklearn.metrics import classification_report
# Printing the classification report comparing the true labels (y_test) and the predicted labels (y_pred)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

         0.0       0.50      0.05      0.10        38
         1.0       0.70      0.98      0.81        85

    accuracy                           0.69       123
   macro avg       0.60      0.51      0.45       123
weighted avg       0.64      0.69      0.59       123



In [35]:
from itertools import product  # Importing the product function from itertools module

param_grid_linear = {           #  Define Parameter grid for linear kernel SVM
    'C': [0.1, 5, 10,50,60,70],
    'kernel': ['linear'],
    'gamma': ['scale', 'auto']
}
param_grid_rbf = {              # Define Parameter grid for Radial Basic Function-RBF
    'C': [0.1, 5, 10,50,60,70],
    'kernel': ['rbf'],
    'gamma': ['scale', 'auto']
}
param_grid_poly = {             # Define Parameters grid for Polynomial kernel
    'C': [0.1, 5, 10,50,60,70],
    'kernel': ['poly'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3, 4]
}

In [36]:
# Importing GridSearchCV from sklearn
from sklearn.model_selection import GridSearchCV

# Assigning SVC model into variables
model=SVC()

# Defining the grid search using GridSearchCV
# - The parameter grid is defined by param_grid_poly
# - 'refit=True' ensures that the best estimator found during the grid search is refitted on the whole dataset
# - 'verbose=2' controls the verbosity of the grid search process (higher values result in more output)
# - 'scoring='f1'' specifies the scoring metric for evaluating the model's performance during grid search
# - 'cv=5' specifies 5-fold cross-validation for evaluating each combination of hyperparameters
grid = GridSearchCV(model,param_grid=param_grid_linear, refit = True, verbose = 2,scoring='f1',cv=5)

# fitting the model for grid search
grid.fit(x_train_smote,y_train_smote)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ..................C=0.1, gamma=scale, kernel=linear; total time=   0.0s
[CV] END ...................C=0.1, gamma=auto, kernel=linear; total time=   0.0s
[CV] END ...................C=0.1, gamma=auto, kernel=linear; total time=   0.0s
[CV] END ...................C=0.1, gamma=auto, kernel=linear; total time=   0.0s
[CV] END ...................C=0.1, gamma=auto, kernel=linear; total time=   0.0s
[CV] END ...................C=0.1, gamma=auto, kernel=linear; total time=   0.0s
[CV] END ....................C=5, gamma=scale, kernel=linear; total time=   0.6s
[CV] END ....................C=5, gamma=scale, k

,estimator,SVC()
,param_grid,"{'C': [0.1, 5, ...], 'gamma': ['scale', 'auto'], 'kernel': ['linear']}"
,scoring,'f1'
,n_jobs,None
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,0.1


In [37]:
print(grid.best_params_)  

{'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}


In [38]:
y_hat=grid.predict(x_test)

In [39]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_hat) #With hyperparameters

0.7642276422764228

In [40]:
accuracy_score(y_test,y_pred) #without hyperparameters

0.6910569105691057